# inference uncertainty

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.

Marginalized plots use Salomon2021; vdM2012 remains a point scenario.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import functools
from pathlib import Path
import torch
from scipy.stats import gaussian_kde, truncnorm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
results_dir = configure_run(PROJECT)
results_dir.mkdir(parents=True, exist_ok=True)

def load_checkpoint_any_device(path, device):
    """pickle's internal tensor deserialization calls torch.load directly,
    bypassing any map_location passed to an outer call -- so we patch the
    default for the duration of the load."""
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load
    return obj

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_posteriors = flam_ckpt["posteriors"]
flam_scalers    = flam_ckpt["scalers"]

tng_ckpt = load_checkpoint_any_device(results_dir / "tng_npe_checkpoint.pkl", device)
tng_posteriors = tng_ckpt["posteriors"]
tng_scalers    = tng_ckpt["scalers"]

BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}

# we only use quantities which are calculated from position and velocities for inference.
# seperation 761 +- 11 kpc Li et al. 2021
# vr,vt van der Marel et al. 2012
# vt contested Salomon et al. 2021 give vt = 82.4 +-31.2 km/s. We run both
# log_mass_ratio: ratio of total bound masses not stellar. Sawala et al. 2023. M_mw = 1.1 +- 0.2
# M_31 = 1.5 +- 0.5 x 10^12 Msun.
# One thing to note though boundsubhalo/totalmass is not equal to observational m200 and does not cancel exactly in a ratio; this is a proxy assumption
LG_OBS, LG_UNC = lg_observation("Salomon2021")



LG_SAFE_BLOCKS = ["core_dynamics"]

# We only use blocks which have defensible values and skip other blocks. Those are only used for simulation comparsions.

print(f"Loaded {len(flam_posteriors)} FLAMINGO combos, {len(tng_posteriors)} TNG blocks.")

In [ ]:
# Marginalized scenario: Salomon2021, using support-respecting approximations.
uncertainty_results = {}
for sim, posteriors, scalers in [("flamingo", flam_posteriors, flam_scalers),
                                ("tng", tng_posteriors, tng_scalers)]:
    for key, posterior in posteriors.items():
        block = key.split("__")[0]
        if block not in LG_SAFE_BLOCKS:
            continue
        feats, h0 = BLOCKS[block], H0_BY_SIM[sim]
        p = sample_lg_point(posterior, scalers[key], feats, LG_OBS, h0=h0, device=device)
        m = sample_lg_marginalized(posterior, scalers[key], feats, LG_OBS, LG_UNC,
                                  h0=h0, device=device)
        tag = f"tng__{key}" if sim == "tng" else key
        uncertainty_results[tag] = {"point": p, "marginalized": m,
            "scenario": "Salomon2021", "run_version": RUN_VERSION,
            "uncertainty_model": "independent physical normals; positive vt/distance; ratio truncated above zero"}
        print(tag, "point median", np.median(p), "marginalized median", np.median(m))
with open(results_dir / "uncertainty_marginalized_results.pkl", "wb") as f:
    pickle.dump(uncertainty_results, f)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
xs = np.linspace(11.5, 14.5, 500)

for ax, key in zip(axes, ["core_dynamics__diff_host", "tng__core_dynamics"]):
    r = uncertainty_results[key]
    for label, samples, color in [("point estimate (no obs. uncertainty)", r["point"], "tab:blue"),
                                    ("uncertainty-marginalized", r["marginalized"], "tab:orange")]:
        kde = gaussian_kde(samples)
        ax.plot(xs, kde(xs), color=color, label=label, lw=2)
    ax.set_title(key)
    ax.set_xlabel(r"$\log_{10} M_{200c}$")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"   # adjust if needed

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i    = sample["g_r"][pi];          gr_j    = sample["g_r"][pj]
    sigma_i = sample["vel_disp"][pi];     sigma_j = sample["vel_disp"][pj]
    kap_i   = sample["kappa_rot"][pi];    kap_j   = sample["kappa_rot"][pj]
    met_i   = met_transform["flamingo_met_dex"][pi]
    met_j   = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i + gr_j) / 2, "gr_diff": np.abs(gr_i - gr_j),
        "met_mean": (met_i + met_j) / 2, "met_diff": np.abs(met_i - met_j),
        "sigma_mean": (sigma_i + sigma_j) / 2, "sigma_diff": np.abs(sigma_i - sigma_j),
        "kappa_mean": (kap_i + kap_j) / 2, "kappa_diff": np.abs(kap_i - kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)

ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + ["log_m200c"])
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

In [ ]:
for key in ["core_dynamics__diff_host"]:
    block_name = key.split("__")[0]
    df = flam_diff[BLOCKS[block_name] + ["log_m200c"]].dropna()
    print(f"{key}: training log_m200c range [{df['log_m200c'].min():.2f}, {df['log_m200c'].max():.2f}], "
          f"5th-95th pct [{df['log_m200c'].quantile(0.05):.2f}, {df['log_m200c'].quantile(0.95):.2f}]")

In [ ]:
for key in ["core_dynamics__diff_host"]:
    samples = uncertainty_results[key]["marginalized"]
    print(f"{key}: marginalized posterior 99.5th percentile = {np.percentile(samples, 99.5):.2f} "
          f"(training data max = 13.52)")

In [ ]:
# M31 transverse velocity: point scenarios and the supported uncertainty approximation.
# vdM2012: 17 km/s point with a one-sided 34.3 km/s bound; no invented normal scale.
# Salomon2021: a positive-truncated normal with underlying location 82.4, scale 31.2.
vt_results = {}
for sim, posteriors, scalers in [("flamingo", flam_posteriors, flam_scalers),
                                ("tng", tng_posteriors, tng_scalers)]:
    for key, posterior in posteriors.items():
        block = key.split("__")[0]
        if block not in LG_SAFE_BLOCKS:
            continue
        for scenario in ("vdM2012", "Salomon2021"):
            obs, unc = lg_observation(scenario)
            feats, h0 = BLOCKS[block], H0_BY_SIM[sim]
            p = sample_lg_point(posterior, scalers[key], feats, obs, h0=h0, device=device)
            m = (sample_lg_marginalized(posterior, scalers[key], feats, obs, unc,
                 h0=h0, device=device) if scenario == "Salomon2021" else None)
            tag = f"tng__{key}" if sim == "tng" else key
            vt_results[f"{tag}__{scenario}"] = {"point": p, "marginalized": m,
                "vt": obs["vt"], "vt_unc": unc["vt"], "run_version": RUN_VERSION,
                "marginalized_status": "available approximation" if m is not None else "no justified vt likelihood"}
            print(tag, scenario, "point", np.median(p), "marginalized",
                  np.median(m) if m is not None else "not computed: likelihood unavailable")
with open(results_dir / "vt_variant_results.pkl", "wb") as f:
    pickle.dump(vt_results, f)